# MEDUSA: SFT & GRPO Training Pipeline (Colab Native)

This notebook acts as your orchestrator to fine-tune `Qwen2.5-1.5B/3B` into an Autonomous Data Engineer.
Everything executes seamlessly within Colab cells so you can monitor live training graphs and progress bars.

In [ ]:
# 1. Install dependencies
!pip install --quiet trl transformers datasets openenv-core peft huggingface_hub

# 2. Clone the Medusa Repo to get the solver scripts
!git clone https://github.com/rampluto/Medusa.git
%cd Medusa

print("Dependencies installed and repo cloned!")

## 1. Authentication
Log into your Hugging Face account to enable model pushing when training finishes.

In [ ]:
from huggingface_hub import login
# Use your WRITE token here
login("hf_YOUR_TOKEN_HERE")

## 2. Fast SFT Synthetic Dataset Generation
We dynamically generate identical data engineering pipelines including `<think>` reasoning traces into a local file.

In [ ]:
!python3 scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl

## 3. Dataset Loading & Smoke Testing
Because we run in Colab, we no longer need to push data to Hugging Face. We simply load the JSONL we just created!

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="medusa_sft_100_episodes.jsonl")
print(f"Loaded {len(ds['train'])} SFT interaction steps.")

assert len(ds['train']) > 1000, "Error: Too few steps produced. Check generation script!"

## 4. Supervised Fine-Tuning (SFT) on Hugging Face Jobs
This section launches SFT on Hugging Face GPU hardware and streams logs here from Colab.

In [ ]:
import os
import re
import subprocess

# Optional: verify auth (requires HF_TOKEN in env, or prior `hf auth login`)
subprocess.run(["hf", "auth", "whoami"], check=False)

# Choose hardware flavor from `hf jobs hardware`
HF_FLAVOR = "a10g-large"   # good default for Qwen 3B LoRA
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
HUB_MODEL_ID = "anubhavkamal/medusa-qwen-sft"
DATASET_PATH = "medusa_sft_100_episodes.jsonl"

job_cmd = [
    "hf", "jobs", "run",
    "pytorch/pytorch:2.3.1-cuda12.1-cudnn8-runtime",
    (
        "bash -lc '"
        "pip install -q -U trl transformers datasets peft accelerate huggingface_hub && "
        "git clone https://github.com/rampluto/Medusa.git && "
        "cd Medusa && "
        "python scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl && "
        f"python train_sft.py --dataset {DATASET_PATH} --model-id {MODEL_ID} "
        "--batch-size 2 --grad-accum 8 --max-seq-len 1024 --num-train-epochs 1 "
        "--push-to-hub "
        f"--hub-model-id {HUB_MODEL_ID}'"
    ),
    "--flavor", HF_FLAVOR,
    "--env", "HF_TOKEN=$HF_TOKEN",
    "--timeout", "12h",
    "--detach",
]

res = subprocess.run(job_cmd, capture_output=True, text=True, check=False)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Failed to submit HF Job")

# Extract job id from output
match = re.search(r"\b([a-z0-9]{8,})\b", res.stdout.lower())
JOB_ID = match.group(1) if match else None
print("Submitted JOB_ID:", JOB_ID)
if not JOB_ID:
    raise RuntimeError("Could not parse JOB_ID from hf jobs run output.")

In [ ]:
# Stream remote Hugging Face Job logs in real time
if not JOB_ID:
    raise RuntimeError("JOB_ID is missing. Run the previous cell first.")

!hf jobs logs $JOB_ID --follow

## 5. GRPO Reinforcement Learning
Tests autonomous interactions with your `openenv` Space to update behavior natively.

In [ ]:
import os, re, subprocess, textwrap

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing in this Colab runtime.")

subprocess.run(["hf", "auth", "whoami"], check=False)

HF_FLAVOR = "a10g-large"
SPACE_REPO_ID = "anubhavkamal/medusa-env"
SFT_ADAPTER_ID = "anubhavkamal/medusa-qwen-sft"
HUB_GRPO_ID = "anubhavkamal/medusa-qwen-grpo"
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
JOB_IMAGE = "pytorch/pytorch:2.3.1-cuda12.1-cudnn8-runtime"

python_payload = textwrap.dedent("""
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("MKL_THREADING_LAYER", "GNU")
os.environ.setdefault("MKL_SERVICE_FORCE_INTEL", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.pop("PYTHONPATH", None)

print("[debug] system python:", sys.executable)

VENV_DIR = Path("/tmp/medusa_grpo_venv")
print("[setup] (re)creating clean isolated venv at", VENV_DIR)
subprocess.check_call([sys.executable, "-m", "venv", "--copies", "--clear", str(VENV_DIR)])

PY  = str(VENV_DIR / "bin" / "python")
PIP = [PY, "-m", "pip"]

def clean_env():
    e = os.environ.copy()
    e["VIRTUAL_ENV"] = str(VENV_DIR)
    e["PATH"] = str(VENV_DIR / "bin") + ":" + e.get("PATH", "")
    e.pop("PYTHONPATH", None)
    e.pop("PYTHONHOME", None)
    return e

ENV = clean_env()

print("[setup] upgrade pip/setuptools/wheel...")
subprocess.check_call(PIP + ["install", "-q", "--upgrade", "pip", "setuptools", "wheel"], env=ENV)

print("[setup] install torch stack (cu121) into venv...")
subprocess.check_call(PIP + [
    "install", "-q", "--upgrade", "--no-cache-dir",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.4.1+cu121",
    "torchvision==0.19.1+cu121",
    "torchaudio==2.4.1+cu121",
], env=ENV)

print("[setup] installing build tools for triton...")
subprocess.check_call([
    "bash", "-lc",
    "apt-get update && apt-get install -y build-essential"
])
os.environ.setdefault("CC", "gcc")
print("[setup] install transformers/trl/datasets/peft...")
subprocess.check_call(PIP + [
    "install", "-q", "--upgrade", "--no-cache-dir",
    "transformers==5.5.0",
    "trl==0.24.0",
    "datasets==4.3.0",
    "peft==0.18.0",
    "accelerate",
    "huggingface_hub",
    "regex",
    "tqdm",
    'bitsandbytes',
    'pydantic',
    'openenv-core'
], env=ENV)

print("[setup] install unsloth + unsloth_zoo (no-deps)...")
subprocess.check_call(PIP + [
    "install", "-q", "--upgrade", "--no-cache-dir",
    "--no-deps",
    "unsloth==2026.4.8",
    "unsloth_zoo==2026.4.9",
], env=ENV)

print("[debug] versions in venv:")
subprocess.check_call([
    PY, "-c",
    "import torch, torchvision, torchaudio; "
    "import transformers, trl, datasets, peft, accelerate, huggingface_hub, regex; "
    "print('torch', torch.__version__); "
    "print('torchvision', torchvision.__version__); "
    "print('torchaudio', torchaudio.__version__); "
    "print('transformers', transformers.__version__); "
    "print('trl', trl.__version__); "
    "print('datasets', datasets.__version__); "
    "print('peft', peft.__version__); "
    "print('accelerate', accelerate.__version__); "
    "print('huggingface_hub', huggingface_hub.__version__); "
    "print('regex', regex.__version__)"
], env=ENV)

print("[debug] preflight unsloth import (with torch._inductor.config pre-import)...")
subprocess.check_call([
    PY, "-c",
    "import importlib, torch; "
    "importlib.import_module('torch._inductor.config'); "
    "import unsloth; from unsloth import FastLanguageModel, PatchFastRL; "
    "print('UNSLOTH_IMPORT_OK')"
], env=ENV)

print("[setup] downloading Space snapshot...")
subprocess.check_call([
    PY, "-c",
    "import os; from huggingface_hub import snapshot_download; "
    "snapshot_download(repo_id='{SPACE_REPO_ID}', repo_type='space', local_dir='Medusa', token=os.environ.get('HF_TOKEN')); "
    "print('SNAPSHOT_OK')"
], env=ENV)

os.chdir("Medusa")
print("Working dir:", os.getcwd())
subprocess.check_call([PY, "-m", "pip", "install", "-e", "."], env=ENV)

TRAIN_ARGS = {TRAIN_ARGS}
print("[train] starting GRPO with args:", TRAIN_ARGS)

# Run the script as __main__ but first import torch._inductor.config
subprocess.check_call([
    PY, "-c",
    "import importlib, runpy, sys; "
    "importlib.import_module('torch._inductor.config'); "
    "sys.argv = {TRAIN_ARGS_REPR}; "
    "runpy.run_path(sys.argv[0], run_name='__main__')"
], env=ENV)

print("GRPO completed successfully")
""").format(
    SPACE_REPO_ID=SPACE_REPO_ID,
    TRAIN_ARGS=repr([
        "trainer/train_medusa_grpo.py",
        "--model-name", BASE_MODEL_ID,
        "--sft-adapter", SFT_ADAPTER_ID,
        "--output-dir", "trainer/medusa-grpo-output",
        "--episodes", "16",
        "--dataset-size", "512",
        "--max-steps", "300",
        "--logging-steps", "1",
        "--save-steps", "50",
        "--reward-log-every", "10",
        "--push-to-hub",
        "--hub-model-id","--no-4bit", HUB_GRPO_ID, "--no-4bit",
    ]),
    TRAIN_ARGS_REPR=repr([
        "trainer/train_medusa_grpo.py",
        "--model-name", BASE_MODEL_ID,
        "--sft-adapter", SFT_ADAPTER_ID,
        "--output-dir", "trainer/medusa-grpo-output",
        "--episodes", "16",
        "--dataset-size", "512",
        "--max-steps", "300",
        "--logging-steps", "1",
        "--save-steps", "50",
        "--reward-log-every", "10",
        "--push-to-hub",
        "--hub-model-id", HUB_GRPO_ID, "--no-4bit",
    ]),
)

job_cmd = [
    "hf", "jobs", "run",
    "--flavor", HF_FLAVOR,
    "--secrets", "HF_TOKEN",
    "--timeout", "12h",
    "--detach",
    JOB_IMAGE,
    "python", "-c", python_payload,
]

res = subprocess.run(job_cmd, capture_output=True, text=True, check=False)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)

if res.returncode != 0:
    raise RuntimeError(f"Failed to submit HF Job (exit={res.returncode})")

combined = (res.stdout or "") + "\n" + (res.stderr or "")
m = re.search(r"\b([a-z0-9]{8,})\b", combined.lower())
JOB_ID = m.group(1) if m else None
print("Submitted JOB_ID:", JOB_ID)
if JOB_ID:
    print("Follow logs with:\n  hf jobs logs", JOB_ID, "--follow")
